# SQL query from table names - Continued

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

## The old Prompt

In [2]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [6]:
context = [{
    "role": "system",
    "content": """
CREATE TABLE guests (
    guest_id INTEGER,
    name TEXT,
    country TEXT
);

INSERT INTO guests VALUES
(1, 'Carlos', 'Spain'),
(2, 'Emma', 'UK'),
(3, 'Yuki', 'Japan'),
(4, 'Ali', 'UAE');


CREATE TABLE properties (
    property_id INTEGER,
    city TEXT,
    price_per_night INTEGER
);

INSERT INTO properties VALUES
(1, 'Tokyo', 120),
(2, 'Madrid', 90),
(3, 'Dubai', 200),
(4, 'Paris', 150);


CREATE TABLE bookings (
    booking_id INTEGER,
    guest_id INTEGER,
    property_id INTEGER,
    nights INTEGER
);

INSERT INTO bookings VALUES
(1, 1, 1, 3),
(2, 2, 2, 2),
(3, 3, 3, 5),
(4, 1, 4, 1),
(5, 4, 3, 4);


CREATE TABLE reviews (
    review_id INTEGER,
    booking_id INTEGER,
    rating INTEGER,
    comment TEXT
);

INSERT INTO reviews VALUES
(1, 1, 5, 'Amazing stay'),
(2, 2, 4, 'Very good'),
(3, 3, 3, 'Okay'),
(4, 4, 5, 'Perfect'),
(5, 5, 2, 'Not great');
"""
}]



In [7]:
context.append({
    "role": "system",
    "content": """
Q: What is the total spent by each customer?
A:
SELECT customer_id, SUM(total_price)
FROM orders
GROUP BY customer_id;

Q: Which customers are from Tokyo?
A:
SELECT * FROM customers
WHERE city = 'Tokyo';
"""
})

In [8]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [13]:
#new
context_user = context.copy()
print(return_CCRMSQL(
    "Show the total money spent by each guest",
    context_user
))

```sql
SELECT g.guest_id, g.name, g.country, SUM(p.price_per_night * b.nights) AS total_spent
FROM guests g
JOIN bookings b ON g.guest_id = b.guest_id
JOIN properties p ON b.property_id = p.property_id
GROUP BY g.guest_id, g.name, g.country;
```


In [14]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("Which city generates the highest total revenue?", old_context_user))

This is your SQL:
```sql
SELECT city, SUM(revenue) AS total_revenue
FROM sales
GROUP BY city
ORDER BY total_revenue DESC
LIMIT 1;
```

This SQL query selects the city and calculates the total revenue generated in each city from the "sales" table. It then groups the results by city, orders them by total revenue in descending order, and limits the output to only show the city with the highest total revenue.


In [15]:
#new
print(return_CCRMSQL("What is the average rating for each property?", context_user))

To calculate the average rating for each property, you can use the following SQL query:

```sql
SELECT p.property_id, p.city, AVG(r.rating) AS average_rating
FROM properties p
JOIN bookings b ON p.property_id = b.property_id
JOIN reviews r ON b.booking_id = r.booking_id
GROUP BY p.property_id, p.city;
```

This query joins the `properties`, `bookings`, and `reviews` tables to calculate the average rating for each property by grouping the results by `property_id` and `city`.


In [16]:
#old
print(return_CCRMSQL("Show guest name, city, total nights stayed, and total money spent", old_context_user))

This is your SQL:
```sql
SELECT guests.name, guests.city, SUM(bookings.nights_stayed) AS total_nights, SUM(bookings.total_cost) AS total_money
FROM guests
JOIN bookings ON guests.guest_id = bookings.guest_id
GROUP BY guests.name, guests.city;
```
This SQL query retrieves the guest's name, city, total nights stayed, and total money spent by joining the "guests" and "bookings" tables on the guest_id. It then calculates the sum of nights stayed and total cost for each guest, grouping the results by guest name and city.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

# Final Report
In this lab, I tested how GPT generates SQL queries from natural language. I found that the results depend a lot on how clear the input is. When the tables and columns were well defined, the model worked much better.

Some variations didn’t work well, especially when the questions were too vague. In those cases, GPT sometimes guessed wrong or used columns that didn’t exist. Also, without examples, the SQL was less accurate and consistent.

What I learned is that prompt design is very important. The more specific and structured the input is, the better the output. GPT is useful, but it still needs clear instructions and should not be fully trusted without checking the results.